# 01 – Sentiment Model from Artifactory

Load a HuggingFace sentiment model via JFrog Artifactory and run predictions.

**Prerequisites:** Set `HF_ENDPOINT` and `HF_TOKEN` (or use `.env`).

In [ ]:
# Load .env if present
import os
from pathlib import Path

try:
    env_file = Path(__file__).parent / ".env"
except NameError:
    env_file = Path(".env")
if not env_file.exists():
    env_file = Path("../.env")
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))

# Check config
hf_endpoint = os.environ.get("HF_ENDPOINT")
hf_token = os.environ.get("HF_TOKEN")
print(f"HF_ENDPOINT: {hf_endpoint or '(not set - will use public HuggingFace)'}")
print(f"HF_TOKEN: {'***' if hf_token else '(not set)'}")

## Load model from Artifactory (or HuggingFace Hub)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_id = os.environ.get("HF_MODEL_ID", "distilbert-base-uncased-finetuned-sst-2-english")

print(f"Loading model: {model_id}")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)
print("Model loaded.")

## Run predictions

In [ ]:
import torch

texts = ["This product is amazing!", "I hate it.", "It's okay."]
inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")

model.eval()
with torch.no_grad():
    outputs = model(**inputs)

probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
scores, indices = torch.max(probs, dim=-1)
id2label = model.config.id2label

for text, idx, score in zip(texts, indices.tolist(), scores.tolist()):
    label = id2label.get(idx, "unknown")
    print(f"  {text!r} -> {label} ({score:.3f})")